##Demonstration: Fine-Tuning a LLM with LoRA

#Scenario

* HelpDesk Lite — a small startup wants a lightweight assistant that writes short customer-support replies (“refund window?”, “late delivery?”). They will fine-tune a tiny local LLM on a few domain examples and compare output quality against the baseline model.

#Objectives

* Prepare a tiny domain dataset

* Apply LoRA (adapter-based tuning) on a small causal LLM

* Build a minimal fine-tuning pipeline

* Evaluate the fine-tuned model vs the baseline on a toy set

A small startup called HelpDesk Lite wants to build a very lightweight AI assistant that can generate short customer-support replies.
These replies are typically simple, such as:

“What is your refund window?”

“My delivery is late — what should I do?”

“How do I cancel an order?”

Because the company is small, it does not want:

Large cloud-based models

Expensive API calls

Heavy GPU training

Instead, the company wants a tiny local LLM, fine-tuned specifically for their domain.
This allows them to:

Run it locally

Pay no per-token fees

Keep customer data private

Customize tone, style, and response format

This scenario is realistic for startups that want AI automation without heavy infrastructure.

## Apply LoRA (adapter-based tuning)

Instead of training the full model (which is expensive), we use LoRA:

Only a small number of trainable weights (adapters)

Fast training on CPU or small GPU

Great results on small datasets

You will fine-tune a small causal language model such as:

distilgpt2

tinyllama

mistral-tiny

or any toy LLM suitable for CPU fine-tuning

LoRA adds domain adaptation without touching full weights.

#Install libraries

In [1]:
!pip install -q transformers peft

#Import and Load Tiny Model

This code initializes a tiny causal language model (Tiny-GPT2) that will be used as the base for LoRA fine-tuning. It first loads the tokenizer for the model and ensures the tokenizer has a valid padding token by falling back to eos_token if needed—this avoids issues during batching. The script then loads the pretrained GPT-2 model (sshleifer/tiny-gpt2), which is extremely small and suitable for fast demonstrations or CPU-based fine-tuning. After loading, the model is moved to the appropriate device (GPU if available, otherwise CPU). This setup prepares the tokenizer and model for the next steps of the LoRA training pipeline, where we will attach LoRA adapters, train on a small dataset, and evaluate the fine-tuned model.

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "openai-community/gpt2"   # very small model for demo
tok = AutoTokenizer.from_pretrained(model_name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

#Prepare Training Data

This section constructs a tiny custom training dataset for fine-tuning. Each example contains a short customer-support “instruction” paired with the ideal “response” the model should learn to generate. The format_example() function converts each example into a simple, consistent text format that the model can learn from, combining an instruction and its response into a single training string. These formatted examples are then tokenized using the previously loaded tokenizer, producing padded input tensors suitable for batching. The labels are set to be the same as the input IDs, which is standard for causal language modeling—meaning the model learns to predict each next token in the combined input text. This prepares the dataset for LoRA fine-tuning in the next step.

In [11]:
train_examples = [
    {"instruction": "Customer asks about refund window", "response": "Our refund window is 30 days from delivery."},
    {"instruction": "Order arrived late", "response": "Sorry for the delay. A delivery credit has been applied."},
    {"instruction": "Wrong item received", "response": "We’ll ship the correct item and provide a return label."},
]

def format_example(ex):
    return f"Instruction: {ex['instruction']}\nResponse: {ex['response']}"

train_texts = [format_example(ex) for ex in train_examples]
inputs = tok(train_texts, padding=True, return_tensors="pt").to(device)
labels = inputs["input_ids"].clone()


#Add LoRA Adapters

This block configures and applies LoRA (Low-Rank Adaptation) to the tiny GPT-2 model, enabling efficient fine-tuning with only a small number of additional trainable parameters. The LoraConfig defines how the adapters should behave: r=4 controls the rank of the low-rank update matrices (making training lightweight), lora_alpha=8 scales the update, and lora_dropout=0.05 adds a small amount of regularization. The target_modules=["c_attn", "c_proj"] selection tells LoRA to inject adapters specifically into the GPT-2 attention projection layers—these are the layers most responsible for generating meaningful responses. Using get_peft_model(), the base model is wrapped with these LoRA adapters, transforming it into a fine-tunable model while keeping the original weights frozen. Finally, the adapted model is moved onto the available device (GPU or CPU). This prepares a highly efficient, low-cost setup for fine-tuning on the small customer-support dataset.

In [12]:
lora_cfg = LoraConfig(
    r=4, lora_alpha=8, target_modules=["c_attn", "c_proj"], lora_dropout=0.05
)
model = get_peft_model(base_model, lora_cfg)
model.to(device)


PeftModel(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora

#Training Loop (One Epoch)

This block performs a very small fine-tuning run on the LoRA-enabled GPT-2 model using the tiny customer-support dataset. An AdamW optimizer is initialized with a relatively high learning rate (1e-3), which is acceptable because LoRA trains only a small number of lightweight adapter parameters. The model is set to training mode, and then a short training loop runs for just three epochs—sufficient for demonstration purposes. In each epoch, gradients are cleared, the model is forward-passed with both input tokens and labels (causal LM training), and the resulting loss is computed. Backpropagation is performed to update only the LoRA adapter weights, after which the optimizer takes a step. The training loss is printed each epoch, giving visibility into how quickly the small model learns the customer-support response patterns. This completes the minimal fine-tuning stage of the pipeline.

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(3):  # few tiny passes
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")


Epoch 1 Loss: 5.9157
Epoch 2 Loss: 5.8880
Epoch 3 Loss: 5.6046


#Compare Baseline vs Fine-Tuned Output

This block defines a helper function, generate_response(), which uses the fine-tuned LoRA-enhanced model to generate customer-support-style replies for new instructions. The function formats each prompt using the same pattern as the training data (“Instruction: … Response:”), tokenizes it, and runs it through the model’s generate() method to predict up to 30 new tokens of output. The generated tokens are then decoded back into readable text.

A small list of evaluation prompts—mirroring the training examples—is provided to test whether the fine-tuned model learned the desired behavior. For each prompt, the script calls generate_response(model, p) and prints the result. Because the model has just been fine-tuned on domain-specific examples, it should produce short, accurate customer-support responses that closely match the style and content of the target training outputs. This section completes the mini end-to-end pipeline by demonstrating how fine-tuning improves the model’s ability to respond correctly in the target domain.

In [10]:
def generate_response(model, prompt):
    inp = tok(f"Instruction: {prompt}\nResponse:", return_tensors="pt").to(device)
    out = model.generate(**inp, max_new_tokens=30, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

prompts = [
    "Customer asks about refund window",
    "Order arrived late",
    "Wrong item received",
]

print("=== Fine-tuned model responses ===")
for p in prompts:
    print(generate_response(base_model, p))


=== Fine-tuned model responses ===
Instruction: Customer asks about refund window
Response: Customer asks about refund window
Customer asks about refund window
Customer asks about refund window
Customer asks about refund window
Customer asks about refund window

Instruction: Order arrived late
Response: No response
Response: No response
Response: No response
Response: No response
Response: No response
Response: No response
Response:
Instruction: Wrong item received
Response:

The following is a list of the items that were not received.

Item Name Description Item Name Description Item Name Description Item Name Description Item
